In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

# 크롬 드라이버 설정
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # 창 안 띄우기
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

BASE_URL = "https://www.card-gorilla.com"
TEAM_URL = f"{BASE_URL}/team/detail/1"

def get_card_links():
    driver.get(TEAM_URL)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select(".card-list .card")
    
    card_info = []
    for card in cards:
        try:
            name = card.select_one(".name").get_text(strip=True)
            link = BASE_URL + card.select_one("a")["href"]
            card_info.append((name, link))
        except:
            continue
    return card_info

def get_card_details(card_name, card_url):
    driver.get(card_url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    # 연회비
    try:
        fee = soup.select_one(".annual_fee").get_text(strip=True)
    except:
        fee = "정보 없음"
    
    # 혜택 목록
    benefit_items = soup.select(".benefit-item")
    benefit_list = []
    for item in benefit_items:
        try:
            b_title = item.select_one(".title").get_text(strip=True)
            b_desc = item.select_one(".desc").get_text(" ", strip=True)
            benefit_list.append((b_title, b_desc))
        except:
            continue
    
    # 데이터 포맷 정리
    data_rows = []
    for b_title, b_desc in benefit_list:
        data_rows.append({
            "카드이름": card_name,
            "연회비": fee,
            "혜택명": b_title,
            "혜택설명": b_desc,
            "상세페이지": card_url
        })
    return data_rows

def main():
    all_cards = get_card_links()
    print(f"[INFO] 카드 {len(all_cards)}개 수집 시작")

    all_data = []

    for i, (name, link) in enumerate(all_cards, start=1):
        print(f"[{i}] {name} 크롤링 중...")
        try:
            details = get_card_details(name, link)
            all_data.extend(details)
        except Exception as e:
            print(f"  오류 발생: {e}")
        time.sleep(1)  # 서버 부하 방지

    # CSV 저장
    df = pd.DataFrame(all_data)
    df.to_csv("card_gorilla_benefits.csv", index=False, encoding="utf-8-sig")
    print("[완료] card_gorilla_benefits.csv 파일로 저장됨")

    driver.quit()

if __name__ == "__main__":
    main()


[INFO] 카드 0개 수집 시작
[완료] card_gorilla_benefits.csv 파일로 저장됨


In [12]:
import requests
from bs4 import BeautifulSoup

url = "https://www.card-gorilla.com/card/detail/2676"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

element = soup.select_one("strong.card")
if element:
    name = element.text.strip()
    print(name)
else:
    print("해당 요소를 찾을 수 없습니다.")

해당 요소를 찾을 수 없습니다.


In [63]:
crawler = CardGorillaCrawler()
test_data = crawler.crawl_range(start_id=2676, end_id=2677, delay_range=(1, 2))
crawler.print_summary()

카드 ID 2676부터 2677까지 크롤링 시작...
크롤링 중: 카드 ID 2676 (1/2)
✅ 성공: 삼성 iD GLOBAL 카드 | 카드고릴라
크롤링 중: 카드 ID 2677 (2/2)
✅ 성공: 카드의정석 Dear, Shopper(디어쇼퍼) | 카드고릴라

크롤링 완료!
성공: 2개, 실패: 0개

=== 크롤링 결과 요약 ===
총 수집된 카드 수: 2

=== 샘플 데이터 (첫 번째 카드) ===
card_id: 2676
url: https://www.card-gorilla.com/card/detail/2676
card_name: 삼성 iD GLOBAL 카드 | 카드고릴라
card_company: 삼성카드
annual_fee: 정보 없음
benefits: 해외 이용 2% 할인, 공항라운지 연 2회 무료 이용, 디지털콘텐츠 50% 할인
requirements: 정보 없음
description: 정보 없음


In [ ]:
import requests

url = "https://api.card-gorilla.com:8080/v1/cards/2677"  # 실제 Request URL로 교체
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.card-gorilla.com",
}

res = requests.get(url, headers=headers)
print(res.text)  # 또는 res.json() if JSON 반환된다면


{"idx":2677,"cid":"677","cate":"CRD","corp":{"idx":5,"name":"\uc6b0\ub9ac\uce74\ub4dc","name_eng":"woori","color":null,"logo_img":{"name":"logo_wr.png","url":"https:\/\/d1c5n4ri2guedi.cloudfront.net\/corp\/5\/logo_img\/33329\/logo_wr.png"},"pr_container":"<p>\uc5f0\ud68c\ube44 \uce90\uc2dc\ubc31 \uc774\ubca4\ud2b8!<\/p>","pr_detail_img":{"name":"samsungbanner.png","url":"https:\/\/d1c5n4ri2guedi.cloudfront.net\/corp\/5\/pr_detail_img\/25412\/samsungbanner.png"},"pr_detail":"<div class=\"pop_box\"><div class=\"close_btn\"><br><\/div><div class=\"banner\"><h3 style=\"background:#0067ab;\">\uc6b0\ub9ac\uce74\ub4dc \uc628\ub77c\uc778 \uc2e0\uaddc\ud68c\uc6d0 \uc5f0\ud68c\ube44 100% \uce90\uc2dc\ubc31 \uc774\ubca4\ud2b8<\/h3><p class=\"title\">\uae30\uac04<\/p><p>2024\ub144 5\uc6d4 1\uc77c(\uc218)~2024\ub144 5\uc6d4 31\uc77c(\uae08)<\/p><p class=\"title\">\ub300\uc0c1<\/p><p class=\"item\">\uc6b0\ub9ac\uce74\ub4dc \uc628\ub77c\uc778 \uc2e0\uaddc\ubc1c\uae09 \ud68c\uc6d0 (\uc6b0\ub9ac\uce74\

In [69]:
res.text.split(",")

['{"idx":2677',
 '"cid":"677"',
 '"cate":"CRD"',
 '"corp":{"idx":5',
 '"name":"\\uc6b0\\ub9ac\\uce74\\ub4dc"',
 '"name_eng":"woori"',
 '"color":null',
 '"logo_img":{"name":"logo_wr.png"',
 '"url":"https:\\/\\/d1c5n4ri2guedi.cloudfront.net\\/corp\\/5\\/logo_img\\/33329\\/logo_wr.png"}',
 '"pr_container":"<p>\\uc5f0\\ud68c\\ube44 \\uce90\\uc2dc\\ubc31 \\uc774\\ubca4\\ud2b8!<\\/p>"',
 '"pr_detail_img":{"name":"samsungbanner.png"',
 '"url":"https:\\/\\/d1c5n4ri2guedi.cloudfront.net\\/corp\\/5\\/pr_detail_img\\/25412\\/samsungbanner.png"}',
 '"pr_detail":"<div class=\\"pop_box\\"><div class=\\"close_btn\\"><br><\\/div><div class=\\"banner\\"><h3 style=\\"background:#0067ab;\\">\\uc6b0\\ub9ac\\uce74\\ub4dc \\uc628\\ub77c\\uc778 \\uc2e0\\uaddc\\ud68c\\uc6d0 \\uc5f0\\ud68c\\ube44 100% \\uce90\\uc2dc\\ubc31 \\uc774\\ubca4\\ud2b8<\\/h3><p class=\\"title\\">\\uae30\\uac04<\\/p><p>2024\\ub144 5\\uc6d4 1\\uc77c(\\uc218)~2024\\ub144 5\\uc6d4 31\\uc77c(\\uae08)<\\/p><p class=\\"title\\">\\ub300\\uc0c

In [ ]:
import requests
import json

url = "https://api.card-gorilla.com:8080/v1/cards/2677"
headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)

# JSON으로 파싱
data = res.json()

# 카드 이름, 카드사, 카테고리 등 추출
card_idx = data.get("idx")
card_name = data.get("name")
card_company = data.get("corp", {}).get("name")
card_category = data.get("cate")


print("카드 ID:", card_idx)
print("카드 이름:", card_name)
print("카드사:", card_company)
print("카테고리:", card_category)


카드 ID: 2677
카드 이름: 카드의정석 Dear, Shopper(디어쇼퍼)
카드사: 우리카드
카테고리: CRD


In [74]:
import requests
import re
import json

url = "https://api.card-gorilla.com:8080/v1/cards/2677"
headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)
data = res.json()

# 카드 이름, 카드사, 카테고리 등 추출
card_idx = data.get("idx")
card_name = data.get("name")
card_company = data.get("corp", {}).get("name")
card_category = data.get("cate")

# 국내/외 연회비 추출
annual_fee_detail = data.get("annual_fee_detail", "")
domestic_fee = None
foreign_fee = None

if annual_fee_detail:
    domestic_match = re.search(r'국내.*?([0-9,]+)원', annual_fee_detail)
    if domestic_match:
        domestic_fee = domestic_match.group(1)
    foreign_match = re.search(r'해외.*?([0-9,]+)원', annual_fee_detail)
    if foreign_match:
        foreign_fee = foreign_match.group(1)

# 카드 혜택 추출
benefits = []
key_benefit = data.get("key_benefit", [])
for benefit in key_benefit:
    title = benefit.get("title")
    desc = benefit.get("desc") or benefit.get("description") or "설명 없음"
    benefits.append({"이름": title, "설명": desc})


# 결과 출력
print("카드 ID:", card_idx)
print("카드 이름:", card_name)
print("카드사:", card_company)
print("카테고리:", card_category)
print("국내 연회비:", domestic_fee)
print("해외 연회비:", foreign_fee)
print("카드 혜택:", benefits)
for idx, benefit in enumerate(benefits, 1):
    print(f"{idx}. {benefit['이름']} - {benefit['설명']}")


카드 ID: 2677
카드 이름: 카드의정석 Dear, Shopper(디어쇼퍼)
카드사: 우리카드
카테고리: CRD
국내 연회비: 150,000
해외 연회비: 150,000
카드 혜택: [{'이름': '바우처', '설명': '설명 없음'}, {'이름': '쇼핑', '설명': '설명 없음'}, {'이름': '적립', '설명': '설명 없음'}, {'이름': '적립', '설명': '설명 없음'}, {'이름': '적립', '설명': '설명 없음'}, {'이름': '공항라운지', '설명': '설명 없음'}, {'이름': '유의사항', '설명': '설명 없음'}]
1. 바우처 - 설명 없음
2. 쇼핑 - 설명 없음
3. 적립 - 설명 없음
4. 적립 - 설명 없음
5. 적립 - 설명 없음
6. 공항라운지 - 설명 없음
7. 유의사항 - 설명 없음


In [ ]:
import requests
import re
import json
from bs4 import BeautifulSoup  # HTML 태그 제거용

url = "https://api.card-gorilla.com:8080/v1/cards/2677"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

# 카드 이름, 카드사, 카테고리 등 추출
card_idx = data.get("idx")
card_name = data.get("name")
card_company = data.get("corp", {}).get("name")
card_category = data.get("cate")

# 국내/외 연회비 추출
annual_fee_detail = data.get("annual_fee_detail", "")
domestic_fee = re.search(r'국내.*?([0-9,]+)원', annual_fee_detail)
domestic_fee = domestic_fee.group(1) if domestic_fee else None
foreign_fee = re.search(r'해외.*?([0-9,]+)원', annual_fee_detail)
foreign_fee = foreign_fee.group(1) if foreign_fee else None

# 카드 혜택(이름과 설명) 추출
benefits = []
key_benefit = data.get("key_benefit", [])
for benefit in key_benefit:
    title = benefit.get("title")
    comment = benefit.get("comment") or ""
    info = benefit.get("info") or ""
    
    # comment가 있으면 comment 사용, 없으면 info의 첫 문장 요약
    if comment:
        desc = comment
    elif info:
        # HTML 태그 제거 및 첫 문장 추출
        soup = BeautifulSoup(info, "html.parser")
        text = soup.get_text().strip()
        desc = text.split("\n")[0].strip() if text else ""
        if len(desc) > 100:  # 너무 길면 앞부분만 자르기
            desc = desc[:100] + "..."
    else:
        desc = "설명 없음"
    
    benefits.append({"이름": title, "설명": desc})

# 결과 출력
print("카드 ID:", card_idx)
print("카드 이름:", card_name)
print("카드사:", card_company)
print("카테고리:", card_category)
print("국내 연회비:", domestic_fee)
print("해외 연회비:", foreign_fee)
print("카드 혜택:")
for idx, benefit in enumerate(benefits, 1):
    print(f"{idx}. {benefit['이름']} - {benefit['설명']}")


카드 ID: 2677
카드 이름: 카드의정석 Dear, Shopper(디어쇼퍼)
카드사: 우리카드
카테고리: CRD
국내 연회비: 150,000
해외 연회비: 150,000
카드 혜택:
1. 바우처 - 프리미엄 기프트 제공(①-④ 중 택 1, 연 1회)
2. 쇼핑 - [퍼스널 쇼핑 적립] 패션/럭셔리/라이프/해외 5% 적립
3. 적립 - [기본 적립] 국내 가맹점 1% 적립
4. 적립 - [FLEX 적립] 국내 가맹점 건당 100만원 이상 결제 시 5% 적립
5. 적립 - [보너스 적립] 본인카드로 국내 가맹점 1천만원 이상 이용 시 5만점 적립
6. 공항라운지 - [더라운지 서비스] 국내외 공항라운지 무료 이용 (본인+동반 1인까지 제공)
7. 유의사항 - 꼭 확인하세요!


In [82]:
import requests
import re
import json
import pandas as pd
from bs4 import BeautifulSoup  # HTML 태그 제거용
from tqdm import tqdm  # 진행 바 표시용

def extract_card_info(card_id):
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {"User-Agent": "Mozilla/5.0"}

    res = requests.get(url, headers=headers)

    # 존재하지 않는 카드인 경우
    if res.status_code != 200:
        return None

    data = res.json()

    # 카드 기본 정보
    card_idx = data.get("idx")
    card_name = data.get("name")
    card_company = data.get("corp", {}).get("name")

    # 국내/해외 연회비 추출
    annual_fee_detail = data.get("annual_fee_detail") or ""
    domestic_fee = re.search(r'국내.*?([0-9,]+)원', annual_fee_detail)
    domestic_fee = domestic_fee.group(1) if domestic_fee else None
    foreign_fee = re.search(r'해외.*?([0-9,]+)원', annual_fee_detail)
    foreign_fee = foreign_fee.group(1) if foreign_fee else None

    # 카드 혜택
    benefits = []
    key_benefit = data.get("key_benefit", [])
    for benefit in key_benefit:
        title = benefit.get("title")
        comment = benefit.get("comment") or ""
        info = benefit.get("info") or ""

        # 설명 추출
        if comment:
            desc = comment
        elif info:
            soup = BeautifulSoup(info, "html.parser")
            text = soup.get_text().strip()
            desc = text.split("\n")[0].strip() if text else ""
            if len(desc) > 100:
                desc = desc[:100] + "..."
        else:
            desc = "설명 없음"
        
        benefits.append(f"{title}: {desc}")

    benefit_str = "\n".join(benefits)

    return {
        "카드ID": card_idx,
        "카드이름": card_name,
        "카드사": card_company,
        "국내연회비": domestic_fee,
        "해외연회비": foreign_fee,
        "주요혜택": benefit_str
    }

# 수집할 카드 ID 범위 지정 (예: 1~3000)
card_infos = []
for card_id in tqdm(range(1, 3000)):
    info = extract_card_info(card_id)
    if info:  # 정상 카드만 저장
        card_infos.append(info)

# CSV로 저장
df = pd.DataFrame(card_infos)
df.to_csv("card_info.csv", index=False, encoding="utf-8-sig")
print("✅ CSV 저장 완료: card_info.csv")

100%|██████████| 2999/2999 [07:00<00:00,  7.12it/s]


✅ CSV 저장 완료: card_info.csv
